# 02 — Data Preparation

Notebook ini mendokumentasikan **Fase 3 CRISP-DM**: transformasi data mentah menjadi
data siap modeling. Logika yang dipakai identik dengan Script A (`pipeline_cron.py`)
melalui fungsi di `src/transformers.py`.

Notebook 01 menjawab “data apa yang saya punya”, notebook 02 menjawab “apa yang terjadi pada data setelah diolah dan apakah hasilnya valid”. Isinya bukti untuk bab Data Preparation: berapa baris dibuang, berapa persen reduksi dedup, dan apakah bobot alert terjaga.

**Tahapan:**
1. Cleaning & flattening
2. Filter agen (blacklist)
3. Feature engineering temporal (`hour_group`)
4. Deduplikasi temporal (time-window)
5. Validasi hasil dan ringkasan reduksi

## 1. Setup & Import

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from config import (
    DATA_RAW_DIR,
    FIGURES_DIR,
    CLUSTER_FEATURES,
    DEDUP_GROUP_COLS,
    TIME_WINDOW_MINUTES,
    HOUR_GROUPS,
)
from src.transformers import (
    read_wazuh_logs,
    flatten_and_clean_logs,
    apply_feature_engineering,
    deduplicate,
)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

print("✓ Setup selesai.")
print(f"  Time window dedup : {TIME_WINDOW_MINUTES} menit")
print(f"  Kunci dedup       : {DEDUP_GROUP_COLS}")
print(f"  Fitur clustering  : {CLUSTER_FEATURES}")

## 2. Cleaning & Flattening

In [ ]:
raw_records = read_wazuh_logs(DATA_RAW_DIR)
df_clean = flatten_and_clean_logs(raw_records)

rows_removed = len(raw_records) - len(df_clean)
pct_removed = (rows_removed / len(raw_records)) * 100 if raw_records else 0

print(f"Record mentah          : {len(raw_records):,}")
print(f"Setelah cleaning       : {len(df_clean):,}")
print(f"Dibuang (null kritis)  : {rows_removed:,} ({pct_removed:.2f}%)")
df_clean.head()

## 3. Filter Agen (Blacklist)

Daftar agen ditampilkan terlebih dahulu. Isi variabel `BLACKLIST` pada sel berikutnya
dengan agen non-target (mesin uji, dll.). **Nilainya harus sama dengan yang diketik
pada prompt Script A** agar hasil notebook dan pipeline konsisten.

In [ ]:
agent_counts = df_clean["agent_name"].value_counts()
print(f"Agen terdeteksi ({len(agent_counts)}):\n")
print(agent_counts.to_string())

In [ ]:
BLACKLIST = []  # contoh: ["test-server", "project-1"]

unknown = [a for a in BLACKLIST if a not in agent_counts.index]
if unknown:
    print(f"PERINGATAN: agen tidak ditemukan, diabaikan: {unknown}")

df_filtered = df_clean[~df_clean["agent_name"].isin(BLACKLIST)].copy()
if df_filtered.empty:
    raise ValueError("Data kosong setelah blacklist.")

print(f"Agen di-blacklist : {BLACKLIST if BLACKLIST else 'tidak ada'}")
print(f"Baris sebelum     : {len(df_clean):,}")
print(f"Baris sesudah     : {len(df_filtered):,}")
print(f"Baris dibuang     : {len(df_clean) - len(df_filtered):,}")

## 4. Feature Engineering Temporal

`hour_group` dibentuk dari jam (WITA): `dini_hari` 00-05, `pagi` 06-11, `siang` 12-17, `malam` 18-23.

In [ ]:
df_fe = apply_feature_engineering(df_filtered)

order = list(HOUR_GROUPS.keys())
hg = df_fe["hour_group"].value_counts().reindex(order).fillna(0).astype(int)
print(hg.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=hg.index, y=hg.values, ax=ax, color="steelblue", order=order)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_title("Distribusi Alert per Hour Group (sebelum dedup)")
ax.set_xlabel("Hour Group")
ax.set_ylabel("Jumlah Alert")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "dp_hour_group.png", dpi=150)
plt.show()

## 5. Deduplikasi Temporal

Log identik pada kombinasi kunci dedup dalam satu jendela waktu dirangkum menjadi satu baris.
Frekuensi kemunculan disimpan di kolom `alert_count`.

In [ ]:
deduped = deduplicate(df_fe)

before, after = len(df_fe), len(deduped)
reduction = (1 - after / before) * 100

print(f"Sebelum deduplikasi : {before:,} baris")
print(f"Setelah deduplikasi : {after:,} pola unik")
print(f"Tingkat reduksi     : {reduction:.2f}%")
deduped.head()

In [ ]:
# Validasi: tidak ada bobot alert yang hilang
total_count = int(deduped["alert_count"].sum())
print(f"Total alert_count : {total_count:,}")
print(f"Baris sebelum     : {before:,}")
print("✓ Valid, bobot alert terjaga." if total_count == before else "✗ TIDAK SESUAI, periksa fungsi deduplicate.")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=["Sebelum", "Sesudah"], y=[before, after], ax=ax, color="steelblue")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_title(f"Volume Data Sebelum vs Sesudah Dedup ({reduction:.1f}% reduksi)")
ax.set_ylabel("Jumlah Baris")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "dp_before_after_dedup.png", dpi=150)
plt.show()

### 5.1 Reduksi per Agen

In [ ]:
by_agent = pd.DataFrame({
    "sebelum": df_fe.groupby("agent_name").size(),
    "sesudah": deduped.groupby("agent_name").size(),
})
by_agent["reduksi (%)"] = ((1 - by_agent["sesudah"] / by_agent["sebelum"]) * 100).round(2)
by_agent.sort_values("sebelum", ascending=False)

### 5.2 Reduksi pada 10 Rule Terbanyak

In [ ]:
by_rule = pd.DataFrame({
    "sebelum": df_fe.groupby("rule_id").size(),
    "sesudah": deduped.groupby("rule_id").size(),
}).sort_values("sebelum", ascending=False).head(10)
by_rule["reduksi (%)"] = ((1 - by_rule["sesudah"] / by_rule["sebelum"]) * 100).round(2)
by_rule

### 5.3 Distribusi `alert_count`

In [ ]:
print(deduped["alert_count"].describe().round(2).to_string())

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(deduped["alert_count"].clip(upper=deduped["alert_count"].quantile(0.99)), bins=50, color="steelblue")
ax.set_yscale("log")
ax.set_title("Distribusi alert_count per Pola (dipotong persentil 99, skala log)")
ax.set_xlabel("alert_count")
ax.set_ylabel("Jumlah pola (log)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "dp_alert_count.png", dpi=150)
plt.show()

## 6. Validasi Kesiapan Fitur Clustering

In [ ]:
missing_cols = [c for c in CLUSTER_FEATURES if c not in deduped.columns]
assert not missing_cols, f"Kolom fitur hilang: {missing_cols}"

audit = pd.DataFrame({
    "nilai_unik": deduped[CLUSTER_FEATURES].nunique(),
    "null": deduped[CLUSTER_FEATURES].isnull().sum(),
})
print(audit.to_string())
print(f"\nKombinasi fitur unik: {len(deduped[CLUSTER_FEATURES].drop_duplicates()):,}")

## 7. Ringkasan Data Preparation

In [ ]:
print("=" * 60)
print("RINGKASAN DATA PREPARATION")
print("=" * 60)
print(f"Record mentah                 : {len(raw_records):,}")
print(f"Setelah cleaning              : {len(df_clean):,}")
print(f"Agen di-blacklist             : {BLACKLIST if BLACKLIST else 'tidak ada'}")
print(f"Setelah blacklist             : {len(df_filtered):,}")
print(f"Setelah dedup {TIME_WINDOW_MINUTES} menit          : {len(deduped):,} pola unik")
print(f"Tingkat reduksi dedup         : {reduction:.2f}%")
print(f"Fitur clustering              : {CLUSTER_FEATURES}")
print("=" * 60)
print("\n✓ Data Preparation selesai. Lanjut ke 03_modeling-clustering.ipynb.")